In [1]:
import pandas as pd

df = pd.read_csv("processed_v2.csv")

print(df.columns)
print(df["Severity_num"].value_counts())

Index(['Latitude', 'Longitude', 'Number_of_Casualties', 'Number_of_Vehicles',
       'Speed_limit', 'Urban_or_Rural_Area', 'Year', 'Month', 'Day', 'Weekday',
       'Hour', 'Is_Rush_Hour', 'Is_Weekend', 'Is_Night', 'Bad_Road_Condition',
       'Bad_Weather', 'Day_Monday', 'Day_Saturday', 'Day_Sunday',
       'Day_Thursday', 'Day_Tuesday', 'Day_Wednesday', 'Season_Spring',
       'Season_Summer', 'Season_Winter', 'Road_Type_One way street',
       'Road_Type_Roundabout', 'Road_Type_Single carriageway',
       'Road_Type_Slip road', 'Vehicle_Group_Goods',
       'Vehicle_Group_Motorcycle', 'Vehicle_Group_Other',
       'Vehicle_Group_Public', 'Vehicle_Group_Vulnerable',
       'Junction_Control_Grouped_No_Junction',
       'Junction_Control_Grouped_Sign_Controlled',
       'Junction_Control_Grouped_Unknown',
       'Junction_Detail_Grouped_No_Junction',
       'Junction_Detail_Grouped_Roundabout',
       'Junction_Detail_Grouped_Simple_Junction', 'Severity_num',
       'Speed_Category_Me

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# df zaten processed_v2.csv'den okundu
y = df["Severity_num"]
X = df.drop(columns=["Severity_num"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "ExtraTrees": ExtraTreesClassifier(n_estimators=400, n_jobs=-1, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42),
    "GBC": GradientBoostingClassifier(random_state=42),
    "LogReg": make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
}

results = []
for name, m in models.items():
    print("▶", name, "başladı")
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    results.append({
        "model": name,
        "macro_f1": f1_score(y_test, pred, average="macro"),
        "weighted_f1": f1_score(y_test, pred, average="weighted"),
    })
    print("✅", name, "bitti")

pd.DataFrame(results).sort_values("macro_f1", ascending=False)


▶ ExtraTrees başladı
✅ ExtraTrees bitti
▶ RandomForest başladı
✅ RandomForest bitti
▶ GBC başladı
✅ GBC bitti
▶ LogReg başladı
✅ LogReg bitti


,model,macro_f1,weighted_f1
0,ExtraTrees,0.323343,0.788670
1,RandomForest,0.313112,0.789642
3,LogReg,0.310001,0.788165
2,GBC,0.309814,0.788733


In [3]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# 1) Model: cost-sensitive (sınıf ağırlıkları)
model_final = ExtraTreesClassifier(
    n_estimators=600,
    random_state=42,
    n_jobs=-1,
    class_weight={0:1, 1:4, 2:12}   # Slight, Serious, Fatal
)

model_final.fit(X_train, y_train)

# 2) Olasılıklar
y_prob = model_final.predict_proba(X_test)

# 3) Threshold ile karar (Fatal ve Serious için)
t_fatal = 0.12
t_serious = 0.25

y_pred = []
for p in y_prob:
    if p[2] > t_fatal:
        y_pred.append(2)
    elif p[1] > t_serious:
        y_pred.append(1)
    else:
        y_pred.append(0)

print("Classification report:")
print(classification_report(y_test, y_pred, digits=4))

print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_test, y_pred))


Classification report:
              precision    recall  f1-score   support

           0     0.8675    0.8633    0.8654     52656
           1     0.1878    0.1863    0.1870      8148
           2     0.0497    0.0695    0.0580       791

    accuracy                         0.7636     61595
   macro avg     0.3683    0.3730    0.3701     61595
weighted avg     0.7670    0.7636    0.7653     61595

Confusion matrix (rows=true, cols=pred):
[[45458  6374   824]
 [ 6403  1518   227]
 [  543   193    55]]


In [5]:
model_final = ExtraTreesClassifier(
    n_estimators=600,
    random_state=42,
    n_jobs=-1,
    class_weight={0:1, 1:4, 2:25}
)

model_final.fit(X_train, y_train)

y_prob = model_final.predict_proba(X_test)

y_pred = []
for p in y_prob:
    if p[2] > 0.06:
        y_pred.append(2)
    elif p[1] > 0.22:
        y_pred.append(1)
    else:
        y_pred.append(0)

print(classification_report(y_test, y_pred, digits=4))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0     0.8736    0.8026    0.8366     52656
           1     0.1807    0.2208    0.1988      8148
           2     0.0395    0.1631    0.0636       791

    accuracy                         0.7174     61595
   macro avg     0.3646    0.3955    0.3663     61595
weighted avg     0.7712    0.7174    0.7423     61595

[[42261  7927  2468]
 [ 5681  1799   668]
 [  435   227   129]]


In [6]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

# X_train, X_test, y_train, y_test zaten hazır

# sınıf ağırlıklarını veriden türetelim (en sağlam yöntem)
class_counts = y_train.value_counts().sort_index()
total = class_counts.sum()
class_weight = total / (len(class_counts) * class_counts)
# örnek: {0: w0, 1: w1, 2: w2}
w_map = class_weight.to_dict()

# her örneğe sample_weight atayalım
sample_weight = y_train.map(w_map).values

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    tree_method="hist",      # hızlı
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train, sample_weight=sample_weight)

y_prob = xgb.predict_proba(X_test)


In [7]:
t_fatal = 0.06
t_serious = 0.22

y_pred = []
for p in y_prob:
    if p[2] > t_fatal:
        y_pred.append(2)
    elif p[1] > t_serious:
        y_pred.append(1)
    else:
        y_pred.append(0)

print(classification_report(y_test, y_pred, digits=4))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0     0.9555    0.0343    0.0661     52656
           1     0.0932    0.0951    0.0942      8148
           2     0.0149    0.9684    0.0294       791

    accuracy                         0.0543     61595
   macro avg     0.3545    0.3659    0.0632     61595
weighted avg     0.8294    0.0543    0.0694     61595

[[ 1804  7517 43335]
 [   80   775  7293]
 [    4    21   766]]


In [8]:
from sklearn.metrics import f1_score

best = None
for tf in np.arange(0.03, 0.16, 0.01):
    for ts in np.arange(0.15, 0.36, 0.01):
        pred = []
        for p in y_prob:
            if p[2] > tf:
                pred.append(2)
            elif p[1] > ts:
                pred.append(1)
            else:
                pred.append(0)
        macro = f1_score(y_test, pred, average="macro")
        if (best is None) or (macro > best[0]):
            best = (macro, tf, ts)

best  # (best_macro_f1, best_t_fatal, best_t_serious)


(0.22440215280710551, 0.15000000000000002, 0.3500000000000002)

In [9]:
import numpy as np
from sklearn.metrics import f1_score, recall_score

best = None

for tf in np.arange(0.03, 0.16, 0.01):
    for ts in np.arange(0.15, 0.41, 0.01):
        pred = []
        for p in y_prob:
            if p[2] > tf:
                pred.append(2)
            elif p[1] > ts:
                pred.append(1)
            else:
                pred.append(0)

        macro = f1_score(y_test, pred, average="macro")
        fatal_rec = recall_score((y_test==2).astype(int), (np.array(pred)==2).astype(int))
        
        score = macro + 0.5 * fatal_rec  # fatal'a ağırlık

        if (best is None) or (score > best[0]):
            best = (score, macro, fatal_rec, tf, ts)

best  # (score, macro_f1, fatal_recall, t_fatal, t_serious)


(0.6761647968963824,
 0.25581081459549754,
 0.8407079646017699,
 0.15000000000000002,
 0.40000000000000024)

In [10]:
xgb = XGBClassifier(
    n_estimators=1200,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=10,
    gamma=0.2,
    reg_lambda=2.0,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)
xgb.fit(X_train, y_train, sample_weight=sample_weight)
y_prob = xgb.predict_proba(X_test)


In [11]:
import numpy as np
from sklearn.metrics import (
    f1_score,
    recall_score,
    accuracy_score,
    precision_score
)

best = None

results = []

for tf in np.arange(0.03, 0.16, 0.01):
    for ts in np.arange(0.15, 0.41, 0.01):

        preds = []
        for p in y_prob:
            if p[2] > tf:
                preds.append(2)
            elif p[1] > ts:
                preds.append(1)
            else:
                preds.append(0)

        preds = np.array(preds)

        macro_f1 = f1_score(y_test, preds, average="macro")
        weighted_f1 = f1_score(y_test, preds, average="weighted")
        acc = accuracy_score(y_test, preds)

        fatal_recall = recall_score(
            (y_test == 2).astype(int),
            (preds == 2).astype(int)
        )

        serious_recall = recall_score(
            (y_test == 1).astype(int),
            (preds == 1).astype(int)
        )

        score = macro_f1 + 0.5 * fatal_recall

        results.append([
            tf, ts,
            macro_f1,
            weighted_f1,
            fatal_recall,
            serious_recall,
            acc,
            score
        ])

        if (best is None) or (score > best[-1]):
            best = [
                tf, ts,
                macro_f1,
                weighted_f1,
                fatal_recall,
                serious_recall,
                acc,
                score
            ]

columns = [
    "t_fatal",
    "t_serious",
    "macro_f1",
    "weighted_f1",
    "fatal_recall",
    "serious_recall",
    "accuracy",
    "combined_score"
]

import pandas as pd
df_results = pd.DataFrame(results, columns=columns)

# En iyi 10 sonucu görelim
df_results.sort_values("combined_score", ascending=False).head(10)


,t_fatal,t_serious,macro_f1,weighted_f1,fatal_recall,serious_recall,accuracy,combined_score
337,0.15,0.40,0.221011,0.439516,0.888748,0.115366,0.314750,0.665385
336,0.15,0.39,0.218504,0.428924,0.888748,0.126780,0.305561,0.662878
311,0.14,0.40,0.209828,0.418292,0.902655,0.102602,0.294780,0.661156
310,0.14,0.39,0.207469,0.407903,0.902655,0.113279,0.286030,0.658797
335,0.15,0.38,0.214330,0.416029,0.888748,0.136721,0.294764,0.658705
309,0.14,0.38,0.203708,0.395573,0.902655,0.122852,0.275996,0.655035
334,0.15,0.37,0.209085,0.399610,0.888748,0.149975,0.281419,0.653459
285,0.13,0.40,0.196781,0.395849,0.911504,0.086524,0.274389,0.652534
284,0.13,0.39,0.194726,0.385780,0.911504,0.096588,0.266125,0.650478
308,0.14,0.37,0.199052,0.380291,0.902655,0.135125,0.263918,0.650380
